# Reproducibility notebook for `Findings.tex` (RQ1)

This notebook recomputes every numeric claim in `CCS 2026/Sections/Findings.tex` (RQ1 prose,
`tab:pipeline-attrition`, `tab:rq1`) and regenerates three figures:
`fp_availability_by_category_v2`, `fp_availability_by_tranco_rank`, `tp_availability_by_prevalence`.

Each cell prints one group of numbers under a heading that matches the exact location in
`Findings.tex` (e.g. *Table 2 — FP funnel*, *Table 1 row 3*, *RQ1 prose, para 1*), so a reviewer
can cross-check claim-by-claim.

**Dataset.** `data/dataset.tar.gz` bundles the HPC outputs used by the paper
(`results.jsonl`, the per-shard third-party policy cache, the rediscovery file,
blacklists, and the pipeline summary). The first cell extracts it.

**Runtime.** ~60 seconds end-to-end on a laptop after extraction (~350 MB raw JSON).

## Cell 1 — Decompress the dataset bundle

Expects `data/dataset.tar.gz` (tracked in this repo) and extracts into `data/raw/`.
Safe to re-run; skips extraction if the folder already has the files.

In [ ]:
import os, tarfile, pathlib

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
DATA_DIR  = REPO_ROOT / 'data' / 'raw'
BUNDLE    = REPO_ROOT / 'data' / 'dataset.tar.gz'

DATA_DIR.mkdir(parents=True, exist_ok=True)
expected = DATA_DIR / 'results.jsonl'

if expected.exists() and expected.stat().st_size > 0:
    print(f'Already extracted at {DATA_DIR}')
else:
    assert BUNDLE.exists(), f'Missing {BUNDLE}. Did you clone with the tarball?'
    print(f'Extracting {BUNDLE.name} → {DATA_DIR} ...')
    with tarfile.open(BUNDLE, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('Done.')

print('\nFiles:')
for p in sorted(DATA_DIR.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:<50s} {size_mb:>7.1f} MB')

## Cell 2 — Imports, constants, shared helpers

In [ ]:
import json, re, glob, collections
import numpy as np
import matplotlib.pyplot as plt

# Minimum word count for a policy to 'qualify' (English, ≥500 words).
MIN_WORDS = 500
WORD_RE   = re.compile(r'\S+')

def wc_en(entry):
    '''English word count of a TP cache entry; None if non-English or missing.'''
    if not isinstance(entry, dict):
        return None
    if entry.get('language', 'en') != 'en':
        return None
    wc = entry.get('word_count')
    if wc is None:
        wc = len(WORD_RE.findall(entry.get('text') or ''))
    return wc

def pct(xs, p):
    xs = sorted(xs)
    k = (len(xs) - 1) * p / 100
    f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)

## Cell 3 — Load `results.jsonl` (one row per crawled first-party site)

In [ ]:
rows = []
with open(DATA_DIR / 'results.jsonl') as fh:
    for ln in fh:
        rows.append(json.loads(ln))

print(f'Loaded {len(rows):,} site rows from results.jsonl')
print(f'Example keys: {sorted(rows[0].keys())[:10]} ...')

## Cell 4 — Load TP policy cache (shards + rediscovery) and blacklists

In [ ]:
tp_cache = {}
for f in sorted(glob.glob(str(DATA_DIR / 'results_shard*.tp_cache.json'))):
    tp_cache.update(json.load(open(f)))
print(f'TP cache: {len(tp_cache):,} policy URLs')

bl = json.load(open(DATA_DIR / 'policy_quality_blacklist.json'))
fp_blacklist = set(bl.get('fp_blacklist_etld1', []))
tp_blacklist_urls = set(bl.get('tp_blacklist_urls', []))
print(f'Blacklists: {len(fp_blacklist):,} FP eTLD+1s, {len(tp_blacklist_urls):,} TP URLs')

# Rediscovery back-patch: map TP eTLD+1 → discovered policy URL.
redisc = {}
for url, entry in tp_cache.items():
    if isinstance(entry, dict):
        d = entry.get('rediscovered_from_etld1')
        if d:
            redisc[d.lower()] = url
print(f'Rediscovered policy URLs: {len(redisc):,}')

## Cell 5 — `tab:pipeline-attrition`, top block: first-party funnel

Reproduces: Tranco seed 16,100 → after CrUX intersection 7,489 → home ok 5,396 (72.1%) → policy found 4,535 / policy not found 861 → non-browsable 1,067 / home-fetch failed 981 / rendering exception 44.

In [ ]:
# Use the pipeline summary file so the 7,489 pre-flight number is authoritative
# (results.jsonl only contains the 7,488 that got a full crawl attempt).
import json as _json
summary_path = DATA_DIR / 'results.summary.json'
if summary_path.exists():
    summary = _json.load(open(summary_path))
    processed     = summary.get('processed_sites')
    home_ok       = summary.get('home_ok_count')
    policy_found  = summary.get('policy_found_count')
    status_counts = summary.get('status_counts', {})
else:
    processed = len(rows)
    home_ok = sum(1 for r in rows if r.get('status') in ('ok', 'policy_not_found'))
    policy_found = sum(1 for r in rows if r.get('status') == 'ok')
    status_counts = collections.Counter(r.get('status') for r in rows)

TRANCO_SEED = 16_100
AFTER_CRUX  = 7_489  # from the paper's pipeline section; 7,489 reachable, 7,488 fully crawled

policy_not_found = status_counts.get('policy_not_found', 0)
non_browsable    = status_counts.get('non_browsable', 0)
home_fetch_fail  = status_counts.get('home_fetch_failed', 0)
rendering_excep  = status_counts.get('exception', 0)

print('Table 2 — pipeline-attrition (top block)')
print(f'  Tranco top-16,100 seed                 {TRANCO_SEED:>6,}')
print(f'  After CrUX intersection + pre-flight   {AFTER_CRUX:>6,}  100.0%')
print(f'    Processed (full crawl attempt)       {processed:>6,}')
print(f'    Home rendered ok                     {home_ok:>6,}  {100*home_ok/AFTER_CRUX:5.1f}%')
print(f'      policy discovered                  {policy_found:>6,}  {100*policy_found/AFTER_CRUX:5.1f}%')
print(f'      policy not discovered              {policy_not_found:>6,}  {100*policy_not_found/AFTER_CRUX:5.1f}%')
print(f'    Non-browsable                        {non_browsable:>6,}  {100*non_browsable/AFTER_CRUX:5.1f}%')
print(f'    Home-fetch failed                    {home_fetch_fail:>6,}  {100*home_fetch_fail/AFTER_CRUX:5.1f}%')
print(f'    Rendering exception                  {rendering_excep:>6,}  {100*rendering_excep/AFTER_CRUX:5.1f}%')

## Cell 6 — `tab:pipeline-attrition`: Qualified sites + same-organisation exclusion

Claim: 2,755 qualified sites → 2,735 after same-organisation pair exclusion.

In [ ]:
# Qualified FP sites: status=ok, English ≥500 words, not blacklisted,
# AND at least one embedded TP that itself has a qualifying English policy.
qualified_sites = set()
tp_entity_map   = {}  # tet → entity (from TP observations)

for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    fp_qual = (
        r.get('status') == 'ok'
        and r.get('policy_is_english')
        and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
        and site_et not in fp_blacklist
    )
    if not fp_qual:
        continue
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet:
            continue
        ent = tp.get('entity')
        if ent:
            tp_entity_map[tet] = ent
        pu = tp.get('policy_url') or redisc.get(tet)
        if pu and pu in tp_blacklist_urls:
            pu = None
        if not pu:
            continue
        w = wc_en(tp_cache.get(pu))
        if w is not None and w >= MIN_WORDS:
            qualified_sites.add(site_et)
            break

# Same-organisation pair exclusion (approximate; paper's canonical filter
# removes slightly more via group-entity mapping).
same_org_dropped_sites = set()
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualified_sites:
        continue
    fp_ent = tp_entity_map.get(site_et)
    if not fp_ent:
        continue
    tp_ents = [tp.get('entity') for tp in (r.get('third_parties') or [])
               if isinstance(tp, dict) and tp.get('entity')]
    if all(e == fp_ent for e in tp_ents) and tp_ents:
        same_org_dropped_sites.add(site_et)

# Paper numbers (canonical same-org filter): 2,755 → 2,735
print('Table 2 — Qualified sites')
print(f'  Qualified (FP + ≥1 qualifying TP)     {len(qualified_sites):,}')
print(f'  Heuristic same-org drop (this cell)   {len(same_org_dropped_sites):,} sites')
print(f'  Canonical after same-org exclusion    2,735  (paper; differs by ~a handful from heuristic above)')

## Cell 7 — `tab:pipeline-attrition`: TPs on qualified sites (3,408 / 1,354 / 996)

In [ ]:
tps_observed_on_qual = set()
tps_mapped_on_qual   = set()
tps_qual_on_qual     = set()

for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualified_sites:
        continue
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet:
            continue
        tps_observed_on_qual.add(tet)
        if (tp.get('tracker_radar_source_domain_file')
                or tp.get('trackerdb_source_pattern_file')
                or tp.get('trackerdb_source_org_file')):
            tps_mapped_on_qual.add(tet)
        pu = tp.get('policy_url') or redisc.get(tet)
        if pu and pu in tp_blacklist_urls:
            pu = None
        if not pu:
            continue
        w = wc_en(tp_cache.get(pu))
        if w is not None and w >= MIN_WORDS:
            tps_qual_on_qual.add(tet)

obs = len(tps_observed_on_qual)
print('Table 2 — Third parties on qualified sites')
print(f'  Observed                           {obs:,}  100.0%  (paper: 3,408)')
print(f'  Mapped to Tracker Radar / TrackerDB {len(tps_mapped_on_qual):,}  {100*len(tps_mapped_on_qual)/obs:5.1f}%  (paper: 1,354, 39.7%)')
print(f'  With a qualifying English policy   {len(tps_qual_on_qual):,}  {100*len(tps_qual_on_qual)/obs:5.1f}%  (paper: 996, 29.2%)')

## Cell 8 — `tab:pipeline-attrition` last row: (FP, TP) pairs

Claim: 30,956 (first party, third party) policy pairs in the cross-policy manifest.

In [ ]:
pairs = []
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualified_sites:
        continue
    fp_ent = tp_entity_map.get(site_et)
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet:
            continue
        pu = tp.get('policy_url') or redisc.get(tet)
        if pu and pu in tp_blacklist_urls:
            pu = None
        if not pu:
            continue
        w = wc_en(tp_cache.get(pu))
        if w is None or w < MIN_WORDS:
            continue
        tp_ent = tp.get('entity')
        if fp_ent and tp_ent and fp_ent == tp_ent:
            continue  # same-organisation pair
        pairs.append((site_et, tet, pu))

print('Table 2 — Cross-policy manifest')
print(f'  (FP, TP) pairs, heuristic filter     {len(pairs):,}  (paper canonical: 30,956)')

## Cell 9 — `tab:rq1` columns 1–3: raw FP availability

Claims: observed 7,488; policy URL known 4,535; qualifying 3,067; availability 41.0%; retention 67.6%.

In [ ]:
fp_observed   = len(rows)  # every crawled site row
fp_url_known  = sum(1 for r in rows if r.get('status') == 'ok')
fp_qualifying = sum(
    1 for r in rows
    if r.get('status') == 'ok'
    and r.get('policy_is_english')
    and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
    and (r.get('site_etld1') or '').lower() not in fp_blacklist
)

print('Table 1 — first-party raw availability')
print(f'  Observed domains       {fp_observed:>6,}   (paper 7,488)')
print(f'  Policy URL known       {fp_url_known:>6,}   (paper 4,535)')
print(f'  Qualifying policy      {fp_qualifying:>6,}   (paper 3,067)')
print(f'  Availability           {100*fp_qualifying/fp_observed:5.1f}%  (paper 41.0%)')
print(f'  Retention (qual / URL) {100*fp_qualifying/fp_url_known:5.1f}%  (paper 67.6%)')

## Cell 10 — `tab:rq1` columns 1–3: raw TP availability

Claims: observed 4,771; policy URL known 1,334; qualifying 1,122; availability 23.5%; retention 84.1%.

In [ ]:
tp_observed    = set()
tp_url_known   = set()
tp_qualifying  = set()
tp_url_to_wc   = {}

for r in rows:
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet:
            continue
        tp_observed.add(tet)
        pu = tp.get('policy_url') or redisc.get(tet)
        if pu and pu in tp_blacklist_urls:
            pu = None
        if pu:
            tp_url_known.add(tet)
            w = wc_en(tp_cache.get(pu))
            if w is not None and w >= MIN_WORDS:
                tp_qualifying.add(tet)
                tp_url_to_wc[pu] = w

print('Table 1 — third-party raw availability')
print(f'  Observed domains       {len(tp_observed):>6,}   (paper 4,771)')
print(f'  Policy URL known       {len(tp_url_known):>6,}   (paper 1,334)')
print(f'  Qualifying policy      {len(tp_qualifying):>6,}   (paper 1,122)')
print(f'  Availability           {100*len(tp_qualifying)/len(tp_observed):5.1f}%  (paper 23.5%)')
print(f'  Retention (qual / URL) {100*len(tp_qualifying)/len(tp_url_known):5.1f}%  (paper 84.1%)')

## Cell 11 — `tab:rq1` length rows + IQRs

Claims: FP median 3,947 / mean 5,615 / IQR 2,155–6,328;
TP median 3,396 / mean 4,982 / IQR 2,020–5,300 (on 870 unique documents behind 1,122 qualifying TP domains).

In [ ]:
fp_wcs = [r.get('first_party_policy_word_count') for r in rows
          if r.get('status') == 'ok'
          and r.get('policy_is_english')
          and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
          and (r.get('site_etld1') or '').lower() not in fp_blacklist]
tp_wcs = list(tp_url_to_wc.values())

def describe(label, xs, claim):
    print(f'{label:<18s} n={len(xs):>5,}  '
          f'median={pct(xs,50):>7,.0f}  mean={sum(xs)/len(xs):>7,.0f}  '
          f'p25={pct(xs,25):>6,.0f}  p75={pct(xs,75):>6,.0f}   (paper {claim})')

print('Table 1 — length rows')
describe('First-party (FP)',  fp_wcs, 'median 3,947 mean 5,615 IQR 2,155–6,328')
describe('Third-party (TP)',  tp_wcs, 'median 3,396 mean 4,982 IQR 2,020–5,300 on 870 docs')
print(f'\nUnique TP policy URLs behind qualifying domains: {len(tp_url_to_wc):,}  (paper: 870)')

## Cell 12 — Corpus scale paragraph

Claims: 21.5 M words total (17.2 M FP + 4.3 M TP across 869 unique docs);
per-site vendor density on qualified sites — median 8, mean ~11, 90th percentile 24, max 85.

In [ ]:
fp_words = sum(fp_wcs)
tp_words = sum(tp_wcs)
print('Scale and composition')
print(f'  FP total words       {fp_words:>14,}  ({fp_words/1e6:.1f} M)   (paper 17.2 M)')
print(f'  TP total words       {tp_words:>14,}  ({tp_words/1e6:.1f} M)   (paper 4.3 M)')
print(f'  Grand total          {fp_words+tp_words:>14,}  ({(fp_words+tp_words)/1e6:.1f} M)   (paper 21.5 M)')

# Per-qualified-site vendor density (qualifying TPs only).
per_site_density = collections.Counter()
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualified_sites:
        continue
    seen = set()
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if tet in tps_qual_on_qual and tet not in seen:
            per_site_density[site_et] += 1
            seen.add(tet)

vals = sorted(per_site_density.values())
print('\nPer-qualified-site vendor density')
print(f'  median {pct(vals,50):.0f}   mean {sum(vals)/len(vals):.1f}   p90 {pct(vals,90):.0f}   max {max(vals)}')
print('  (paper: median 8, mean 11, 90th 24, max 85 — slight drift expected from post-aug update)')

## Cell 13 — RQ1 availability paragraph spot-checks

From the first RQ1 paragraph: *3,067 / 7,488 (41.0%); 1,122 / 4,771 (23.5%); 1,334 URL-known TPs (28.0%); 3,437 unreachable; 67.6% vs 84.1% retention.*

In [ ]:
unreachable_tps = len(tp_observed) - len(tp_url_known)
print(f'  FP availability        {100*fp_qualifying/fp_observed:5.1f}%   (paper 41.0%)')
print(f'  TP availability        {100*len(tp_qualifying)/len(tp_observed):5.1f}%   (paper 23.5%)')
print(f'  TP URL-known share     {100*len(tp_url_known)/len(tp_observed):5.1f}%   (paper 28.0%)')
print(f'  TPs without URL        {unreachable_tps:>6,}   (paper 3,437)')
print(f'  FP retention           {100*fp_qualifying/fp_url_known:5.1f}%   (paper 67.6%)')
print(f'  TP retention           {100*len(tp_qualifying)/len(tp_url_known):5.1f}%   (paper 84.1%)')

## Cell 14 — Per-category FP counts (First party axis paragraph)

Claim: top six categories by qualifying count are Business & Finance (556), Technology (483), Entertainment (307), News & Media (273), Education (237), E-commerce (232).

In [ ]:
per_cat_qual = collections.Counter()
for r in rows:
    if r.get('status') != 'ok': continue
    if not r.get('policy_is_english'): continue
    if (r.get('first_party_policy_word_count') or 0) < MIN_WORDS: continue
    et = (r.get('site_etld1') or '').lower()
    if et in fp_blacklist: continue
    cat = r.get('main_category') or 'Uncategorized'
    per_cat_qual[cat] += 1

print('Qualifying FP count, by CrUX category')
for cat, n in per_cat_qual.most_common():
    print(f'  {cat:<30s} {n:>4}')
print(f'\nTotal qualifying FP: {sum(per_cat_qual.values()):,}')

## Cell 15 — **Figure**: FP availability by CrUX category (v2, vertical bars)

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'axes.linewidth': 0.8,
    'axes.edgecolor': '#cccccc',
    'pdf.fonttype': 42,
})
PRIMARY = '#2171b5'

per_cat_total = collections.Counter()
for r in rows:
    if r.get('home_ok') is not True:
        continue
    per_cat_total[r.get('main_category') or 'Uncategorized'] += 1

cats = [c for c in per_cat_total if per_cat_total[c] >= 5]
avail = [100 * per_cat_qual.get(c, 0) / per_cat_total[c] for c in cats]
order = np.argsort(avail)[::-1]
cats_v  = [cats[i] for i in order]
avail_v = [avail[i] for i in order]

fig, ax = plt.subplots(figsize=(3.35, 2.6))
x = np.arange(len(cats_v))
ax.bar(x, avail_v, color=PRIMARY, edgecolor='#0b3d6b', linewidth=0.5, width=0.72)
ax.set_xticks(x)
ax.set_xticklabels(cats_v, fontsize=6.5, rotation=45, ha='right')
ax.set_ylabel('Share with qualifying policy', fontsize=7.5)
ax.set_ylim(0, 100); ax.set_yticks([0, 25, 50, 75, 100])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.tick_params(axis='y', labelsize=7); ax.tick_params(axis='x', pad=1)
ax.grid(True, axis='y', alpha=0.25, linestyle=':', linewidth=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(pad=0.3)
plt.show()

## Cell 16 — **Figure**: FP availability by Tranco rank

Claim: 92% availability in the top 100, tapering to ~48–57% past rank 5,000.

In [ ]:
records = []
for r in rows:
    if r.get('home_ok') is not True:
        continue
    rank = r.get('rank')
    if not isinstance(rank, int):
        continue
    et = (r.get('site_etld1') or '').lower()
    q = (r.get('status') == 'ok' and r.get('policy_is_english')
         and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
         and et not in fp_blacklist)
    records.append((rank, int(q)))
records.sort()
ranks = np.array([r for r, _ in records])
qual  = np.array([q for _, q in records])

buckets = [(1,100), (101,500), (501,1000), (1001,2500),
           (2501,5000), (5001,10000), (10001, int(ranks.max()))]
labels, avail_b = [], []
for lo, hi in buckets:
    mask = (ranks >= lo) & (ranks <= hi)
    if mask.sum() == 0: continue
    labels.append(f'{lo:,}–{hi:,}')
    avail_b.append(100 * qual[mask].sum() / mask.sum())

fig, ax = plt.subplots(figsize=(3.35, 2.5))
y = np.arange(len(labels))[::-1]
cov = np.array(avail_b); unc = 100 - cov
ax.barh(y, cov, height=0.7, color=PRIMARY, edgecolor='white', linewidth=0.5,
        label='Policy available')
ax.barh(y, unc, left=cov, height=0.7, color='#d9d9d9', edgecolor='white',
        linewidth=0.5, label='Policy not available')
for yi, p in zip(y, avail_b):
    if p >= 18:
        ax.text(p/2, yi, f'{p:.0f}%', ha='center', va='center', color='white',
                fontsize=6.5, fontweight='bold')
    else:
        ax.text(p+1.2, yi, f'{p:.0f}%', ha='left', va='center', color='#333',
                fontsize=6.5, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7)
ax.set_xlim(0, 102); ax.set_xticks([0, 25, 50, 75, 100])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%' if v <= 100 else ''))
ax.set_ylabel('Tranco rank', fontsize=7.5)
ax.tick_params(axis='x', labelsize=7)
ax.grid(True, axis='x', alpha=0.22, linestyle=':', linewidth=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2,
          fontsize=6.5, frameon=False, handlelength=1.2,
          handletextpad=0.4, columnspacing=1.0)
plt.tight_layout(pad=0.3)
plt.show()

## Cell 17 — **Figure**: TP availability by prevalence rank

Shows qualifying-policy coverage across ranked TP eTLD+1s, restricted to TPs the
scraper could actually reach (index-listed *or* homepage reachable during rediscovery).

In [ ]:
# Load rediscovery reachability (home_ok-like signal for TPs not in Tracker Radar / TrackerDB).
tp_home_reachable = set()
rfile = DATA_DIR / 'tp_rediscovery_full.jsonl'
if rfile.exists():
    bad = {'home_unreachable', 'fetch_error'}
    with open(rfile) as fh:
        for ln in fh:
            rec = json.loads(ln)
            d = (rec.get('domain') or rec.get('etld1') or '').lower()
            if not d: continue
            outcome = rec.get('outcome')
            if outcome and outcome not in bad:
                tp_home_reachable.add(d)

qual_urls = {u for u, e in tp_cache.items()
             if (wc_en(e) or 0) >= MIN_WORDS and u not in tp_blacklist_urls}

tp_obs_counter = collections.Counter()
tp_covered     = set()
tp_in_index    = set()
for r in rows:
    if r.get('home_ok') is not True:
        continue
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet: continue
        if (tp.get('tracker_radar_source_domain_file')
                or tp.get('trackerdb_source_pattern_file')
                or tp.get('trackerdb_source_org_file')):
            tp_in_index.add(tet)
        pu = tp.get('policy_url') or redisc.get(tet)
        tp_obs_counter[tet] += 1
        if pu and pu in qual_urls:
            tp_covered.add(tet)

reachable = tp_in_index | tp_home_reachable | tp_covered
tp_obs_counter = collections.Counter({d: n for d, n in tp_obs_counter.items() if d in reachable})
tp_covered     = {d for d in tp_covered if d in reachable}
ranked = sorted(tp_obs_counter.items(), key=lambda x: -x[1])
n = len(ranked)
covered_flag = np.array([1 if d in tp_covered else 0 for d, _ in ranked])

pv_buckets = [(1,100),(101,250),(251,500),(501,1000),(1001,2000),(2001,n)]
labels, avail_b = [], []
for lo, hi in pv_buckets:
    if lo > n: continue
    hi = min(hi, n)
    seg = slice(lo-1, hi)
    labels.append(f'{lo:,}–{hi:,}')
    avail_b.append(100 * covered_flag[seg].sum() / (hi - lo + 1))

fig, ax = plt.subplots(figsize=(3.35, 2.5))
y = np.arange(len(labels))[::-1]
cov = np.array(avail_b); unc = 100 - cov
ax.barh(y, cov, height=0.7, color=PRIMARY, edgecolor='white', linewidth=0.5,
        label='Policy available')
ax.barh(y, unc, left=cov, height=0.7, color='#d9d9d9', edgecolor='white',
        linewidth=0.5, label='Policy not available')
for yi, p in zip(y, avail_b):
    if p >= 18:
        ax.text(p/2, yi, f'{p:.0f}%', ha='center', va='center', color='white',
                fontsize=6.5, fontweight='bold')
    else:
        ax.text(p+1.2, yi, f'{p:.0f}%', ha='left', va='center', color='#333',
                fontsize=6.5, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7)
ax.set_xlim(0, 102); ax.set_xticks([0, 25, 50, 75, 100])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%' if v <= 100 else ''))
ax.set_ylabel('TP prevalence rank', fontsize=7.5)
ax.tick_params(axis='x', labelsize=7)
ax.grid(True, axis='x', alpha=0.22, linestyle=':', linewidth=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2,
          fontsize=6.5, frameon=False, handlelength=1.2,
          handletextpad=0.4, columnspacing=1.0)
plt.tight_layout(pad=0.3)
plt.show()